# Part A: Profiling, Cleaning, and the Data Story

This notebook contains the complete Exploratory Data Analysis (EDA) pipeline on Seaborn's Titanic dataset.

## 1. Data Ingestion & Offline Fallback Setup

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

CSV_PATH = "titanic.csv"

if not os.path.exists(CSV_PATH):
    print("🌐 Fetching Titanic dataset from Seaborn network cache...")
    df_raw = sns.load_dataset('titanic')
    df_raw.to_csv(CSV_PATH, index=False)
    print(f"✅ Raw dataset successfully saved to offline fallback: {CSV_PATH}")
else:
    print(f"📁 Loading dataset directly from offline file: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
print(f"\nData successfully loaded! Shape: {df.shape}")
df.head()

## 2. Profiling & Missing Values Analysis

In [ ]:
print(f"📊 DATASET SHAPE: {df.shape}\n")
df.info()

print("\n📈 DATASET STATISTICAL SUMMARY:")
display(df.describe(include='all'))

print("\n⚠️ MISSING VALUES (Percentage per column):")
missing_percentages = (df.isnull().sum() / len(df)) * 100
missing_cols = missing_percentages[missing_percentages > 0].sort_values(ascending=False)

for col, pct in missing_cols.items():
    print(f" - '{col}': {pct:.2f}% missing")

### Missing Values Strategy Justification
- **`embarked` & `embark_town`** (0.22% missing): Dropped the rows containing missing values as the missingness is well under the 5% threshold.
- **`age`** (19.87% missing): Imputed missing values using the median age of the passengers, as the missingness is between 5% and 30%.
- **`deck`** (77.22% missing): Dropped the column entirely because a missing rate of over 70% makes imputation highly unreliable.

In [ ]:
df.dropna(subset=['embarked', 'embark_town'], inplace=True)
median_age = df['age'].median()
df['age'] = df['age'].fillna(median_age)
df.drop(columns=['deck'], inplace=True)

print("✅ Missing values successfully handled!")
print(f"📊 Cleaned Dataset Shape: {df.shape}")
print(f"⚠️ Remaining missing values: {df.isnull().sum().sum()}")

## 3. Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['age'], kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Age Distribution (Histogram)')
sns.boxplot(x=df['age'], ax=axes[0, 1], color='lightgreen')
axes[0, 1].set_title('Age Box Plot (Outliers)')

sns.histplot(df['fare'], kde=True, ax=axes[1, 0], color='salmon')
axes[1, 0].set_title('Fare Distribution (Histogram)')
sns.boxplot(x=df['fare'], ax=axes[1, 1], color='gold')
axes[1, 1].set_title('Fare Box Plot (Outliers)')

plt.tight_layout()
plt.show()

In [ ]:
def compute_iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    return len(outliers), lower_bound, upper_bound

age_outliers, age_low, age_high = compute_iqr_outliers(df['age'])
fare_outliers, fare_low, fare_high = compute_iqr_outliers(df['fare'])

print(f"📌 Outliers: 'age' = {age_outliers}, 'fare' = {fare_outliers}")
print(f"📌 Fare Stats: Mean={df['fare'].mean():.4f}, Median={df['fare'].median():.4f}, Mode={df['fare'].mode()[0]:.4f}")

## 4. Bivariate Analysis

In [ ]:
print("Survival Rate by Sex:")
print(df.groupby('sex')['survived'].mean())

print("\nSurvival Rate by Pclass:")
print(df.groupby('pclass')['survived'].mean())

print("\nSurvival Rate by Sex & Pclass:")
for sex in ['female', 'male']:
    for pclass in [1, 2, 3]:
        mask = (df['sex'] == sex) & (df['pclass'] == pclass)
        rate = df[mask]['survived'].mean()
        print(f" - {sex.title()}, Pclass {pclass}: {rate:.4f}")

In [ ]:
corr_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".4f", vmin=-1, vmax=1, square=True)
plt.title('6x6 Correlation Heatmap')
plt.show()

## 5. Multivariate Data Story

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.barplot(x="pclass", y="survived", hue="sex", data=df, ax=axes[0, 0], errorbar=None, palette="muted")
axes[0, 0].set_title("1. Survival Rate by Passenger Class & Sex")

sns.violinplot(x="survived", y="age", hue="sex", data=df, split=True, ax=axes[0, 1], palette="pastel")
axes[0, 1].set_title("2. Age Distribution by Survival Status & Sex")
axes[0, 1].set_xticklabels(["Died", "Survived"])

sns.boxplot(x="survived", y="fare", data=df, ax=axes[1, 0], palette="coolwarm", showfliers=False)
axes[1, 0].set_title("3. Fare Distribution of Survivors vs Non-Survivors")
axes[1, 0].set_xticklabels(["Died", "Survived"])

df_temp = df.copy()
df_temp['family_size'] = df_temp['sibsp'] + df_temp['parch'] + 1
sns.barplot(x="family_size", y="survived", data=df_temp, ax=axes[1, 1], errorbar=None, color="salmon")
axes[1, 1].set_title("4. Survival Rate by Family Size")

plt.tight_layout()
plt.show()

## 6. Exploratory Standardization Check

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[['age', 'fare']] = scaler.fit_transform(df[['age', 'fare']])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['age'], kde=True, color='skyblue', ax=axes[0, 0])
axes[0, 0].set_title("Age Distribution (Before)")

sns.histplot(df_scaled['age'], kde=True, color='blue', ax=axes[0, 1])
axes[0, 1].set_title("Age Distribution (After)")

sns.histplot(df['fare'], kde=True, color='salmon', ax=axes[1, 0])
axes[1, 0].set_title("Fare Distribution (Before)")

sns.histplot(df_scaled['fare'], kde=True, color='red', ax=axes[1, 1])
axes[1, 1].set_title("Fare Distribution (After)")

plt.tight_layout()
plt.show()

print("Original stats:\n", df[['age', 'fare']].describe().loc[['mean', 'std']])
print("\nScaled stats:\n", df_scaled[['age', 'fare']].describe().loc[['mean', 'std']])